# Comprehensive Metaphor Identification Workflow

This notebook integrates dataset loading, prompting, fine-tuning, RAG, and evaluation across OpenAI and Ollama models.

## Section 0: Setup and Dependencies

In [ ]:
!pip install pandas openai transformers peft datasets torch langchain scikit-learn paired -q

In [ ]:
import pandas as pd
import numpy as np
import json
import copy
import re
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

## Section 1: Dataset Loader

Load the metaphor dataset and analyze its structure.

In [ ]:
# Dataset loading function
def load_dataset(filepath='metaphor_dataset.csv'):
    """
    Load the metaphor dataset.
    
    Args:
        filepath: Path to the CSV file
    
    Returns:
        DataFrame with metaphor data
    """
    df = pd.read_csv(filepath)
    return df

# Load dataset
dataset = load_dataset('metaphor_dataset.csv')
print(f'Dataset loaded successfully!')
print(f'Shape: {dataset.shape}')

## Section 2: Comprehensive Dataset Analysis

Analyze the structure and content of the dataset.

In [ ]:
# Dataset facts
print('='*60)
print('DATASET FACTS')
print('='*60)
print(f'Total number of entries: {len(dataset)}')
print(f'Number of columns: {len(dataset.columns)}')
print(f'\nColumn names and types:')
print(dataset.dtypes)
print(f'\nMissing values:')
print(dataset.isnull().sum())
print(f'\nDataset shape: {dataset.shape}')
print(f'\nFirst few entries:')
print(dataset.head(2))

## Section 3: Prompting with OpenAI

**STEP (3)**: Use the dataset for prompting with OpenAI models.

⚠️ **Note**: You need to set your OpenAI API key below.

In [ ]:
# Set your OpenAI API key
my_api_key = 'INSERT YOUR KEY HERE'
# Example: my_api_key = 'sk-proj-xxxxx...'

# Only proceed if API key is set
if my_api_key != 'INSERT YOUR KEY HERE':
    from openai import OpenAI
    client_openai = OpenAI(api_key=my_api_key)
    print('✓ OpenAI client initialized')
else:
    print('⚠️  Please set your OpenAI API key to proceed with OpenAI tasks')
    client_openai = None

In [ ]:
# Prompting function for OpenAI
def prompt_openai(client, text, model='gpt-4.1-mini-2025-04-14'):
    """
    Send a prompting request to OpenAI.
    
    Args:
        client: OpenAI client
        text: Plain text to identify metaphors
        model: Model ID
    
    Returns:
        Tagged text with metaphors
    """
    messages = [
        {'role': 'system',
         'content': 'You are a linguistic expert trained in metaphor identification. When the user provides a text, follow this protocol:\n• Identify all metaphorical expressions.\n• Wrap each one in <Metaphor></Metaphor> tags.\n• Reproduce the rest of the text exactly as written.\n• Do not include any explanation, commentary, or extra content in this message.'},
        {'role': 'user',
         'content': f'Can you please identify and tag the metaphors in the following text?\n{text}'}
    ]
    try:
        response = client.chat.completions.create(model=model, messages=messages, n=1)
        return response.choices[0].message.content
    except Exception as e:
        print(f'Error: {e}')
        return None

# Test on first example
if client_openai is not None:
    test_text = dataset.iloc[0]['plain']
    print('Input text:')
    print(test_text[:200] + '...' if len(test_text) > 200 else test_text)
    print('\nProcessing with OpenAI...')
    # Uncomment below to run (requires API key and will incur costs)
    # result_openai_prompt = prompt_openai(client_openai, test_text)
    # print('\nOutput:')
    # print(result_openai_prompt)
    print('(Skipped - uncomment to run with valid API key)')

## Section 4: Fine-Tuning with OpenAI

**STEP (4)**: Fine-tune OpenAI model on the dataset.

⚠️ **Note**: Fine-tuning requires valid OpenAI API key and incurs costs.

In [ ]:
def prepare_finetuning_data_openai(dataset, train_ratio=0.8, seed=1):
    """
    Prepare data for OpenAI fine-tuning (JSONL format).
    
    Args:
        dataset: DataFrame with 'plain' and tagged text columns
        train_ratio: Train/test split ratio
        seed: Random seed
    
    Returns:
        train_df, test_df, jsonl_path
    """
    # Split data
    train_df = dataset.sample(frac=train_ratio, random_state=seed)
    test_df = dataset.drop(index=train_df.index)
    
    # Prepare JSONL
    user_msg = 'Can you please identify and tag the metaphors in the following text? '
    json_lines = []
    
    for idx in range(len(train_df)):
        raw_text = train_df.iloc[idx]['plain'].replace('\\n', ' ')
        tagged_text = train_df.iloc[idx]['metaphor_tagged_text'].replace('\\n', ' ')
        this_chat = {
            'messages': [
                {'role': 'user', 'content': user_msg + '\n' + raw_text},
                {'role': 'assistant', 'content': tagged_text}
            ]
        }
        json_lines.append(json.dumps(this_chat))
    
    jsonl_path = f'ft_tr{train_ratio}_s{seed}.jsonl'
    with open(jsonl_path, 'w') as f:
        f.write('\n'.join(json_lines))
    
    return train_df, test_df, jsonl_path

# Prepare fine-tuning data
train_df_ft, test_df_ft, jsonl_path = prepare_finetuning_data_openai(dataset)
print(f'Fine-tuning data prepared:')
print(f'  Train set: {len(train_df_ft)} samples')
print(f'  Test set: {len(test_df_ft)} samples')
print(f'  JSONL file: {jsonl_path}')

In [ ]:
# Fine-tuning with OpenAI (requires API key)
# Uncomment and configure as needed

# if client_openai is not None:
#     # Upload file
#     with open(jsonl_path, 'rb') as f:
#         fileinfo = client_openai.files.create(file=f, purpose='fine-tune')
#     
#     # Create fine-tuning job
#     ft_job = client_openai.fine_tuning.jobs.create(
#         training_file=fileinfo.id,
#         model='gpt-4.1-mini-2025-04-14',
#         suffix='metaphor_ft'
#     )
#     print(f'Fine-tuning job created: {ft_job.id}')
#     ft_model_id = None  # Will be available after training completes

print('Fine-tuning setup ready (uncomment above to execute)')

## Section 5: RAG (Retrieval-Augmented Generation) with OpenAI

**STEP (5)**: Use RAG approach with OpenAI.

RAG combines document retrieval with generation for better context-aware responses.

In [ ]:
def prepare_rag_context():
    """
    Prepare RAG context from metaphor protocol.
    """
    rag_context = """METAPHOR IDENTIFICATION PROTOCOL:
    
    A metaphor is a figure of speech that directly compares two different things 
    without using 'like' or 'as'. It implies that something IS something else.
    
    Key characteristics:
    1. Two distinct entities being compared
    2. Implicit comparison (no 'like' or 'as')
    3. Creates new meaning through the comparison
    
    Examples:
    - 'Time is money' (time and money are compared)
    - 'Life is a journey' (life and journey)
    - 'The world is a stage' (world and stage)
    
    When identifying metaphors, tag them with <Metaphor></Metaphor> tags.
    """
    return rag_context

rag_context = prepare_rag_context()
print('RAG context prepared')
print(f'Context length: {len(rag_context)} characters')

In [ ]:
def prompt_openai_rag(client, text, context, model='gpt-4.1-mini-2025-04-14'):
    """
    RAG-based prompting with OpenAI.
    """
    messages = [
        {'role': 'system',
         'content': 'You are a helpful AI assistant. Use the following pieces of context to answer the question at the end. If you don\'t know the answer, just say you don\'t know. DO NOT try to make up an answer.\n' + context},
        {'role': 'system',
         'content': 'You are a linguistic expert trained in metaphor identification. When the user provides a text, follow this protocol:\n• Identify all metaphorical expressions.\n• Wrap each one in <Metaphor></Metaphor> tags.\n• Reproduce the rest of the text exactly as written.\n• Do not include any explanation, commentary, or extra content.'},
        {'role': 'user',
         'content': f'Can you please identify and tag the metaphors in the following text?\n{text}'}
    ]
    try:
        response = client.chat.completions.create(model=model, messages=messages, n=1)
        return response.choices[0].message.content
    except Exception as e:
        print(f'Error: {e}')
        return None

# Test RAG on first example
if client_openai is not None:
    test_text_rag = dataset.iloc[0]['plain']
    print('Testing RAG approach...')
    # Uncomment to run
    # result_openai_rag = prompt_openai_rag(client_openai, test_text_rag, rag_context)
    # print('RAG Result:' , result_openai_rag[:300])
    print('(Skipped - uncomment to run with valid API key)')

## Section 6: Ollama Setup Guide

**STEP (6)**: Installation and setup guide for Ollama.

### Installation Steps:

1. **Download Ollama**:
   - Visit https://ollama.com/
   - Download for your OS (macOS, Linux, Windows)

2. **Install Ollama**:
   - Follow the installation wizard
   - Verify installation: `ollama --version`

3. **Start Ollama Service**:
   - macOS/Linux: `ollama serve`
   - Windows: Ollama runs in the background

4. **Pull Models** (in a separate terminal):
   ```bash
   ollama pull llama3.2:1b
   ollama pull llama3.2:3b
   ollama pull llama3.1:8b
   ```

5. **Verify Setup**:
   ```bash
   ollama list
   ```

In [ ]:
# Check if Ollama is available
try:
    from ollama import chat
    ollama_available = True
    print('✓ Ollama Python package is available')
except ImportError:
    ollama_available = False
    print('⚠️  Ollama package not found. Install with: pip install ollama')

print('\nMake sure Ollama service is running!')
print('Run in terminal: ollama serve')

## Section 7: Prompting with Ollama

**STEP (7)**: Use local Ollama model for metaphor identification.

⚠️ **Note**: Requires Ollama service running and model downloaded.

In [ ]:
def prompt_ollama(text, model='llama3.2:1b'):
    """
    Send a prompting request to Ollama.
    
    Args:
        text: Plain text to identify metaphors
        model: Model ID
    
    Returns:
        Tagged text with metaphors
    """
    messages = [
        {'role': 'system',
         'content': 'You are a linguistic expert trained in metaphor identification. When the user provides a text, follow this protocol:\n• Identify all metaphorical expressions.\n• Wrap each one in <Metaphor></Metaphor> tags.\n• Reproduce the rest of the text exactly as written.\n• Do not include any explanation, commentary, or extra content in this message.'},
        {'role': 'user',
         'content': f'Can you please identify and tag the metaphors in the following text?\n{text}'}
    ]
    try:
        if ollama_available:
            response = chat(model=model, messages=messages)
            return response.message.content
    except Exception as e:
        print(f'Error connecting to Ollama: {e}')
        print('Make sure Ollama service is running!')
    return None

# Test on first example
if ollama_available:
    test_text = dataset.iloc[0]['plain']
    print('Testing Ollama (llama3.2:1b)...')
    print('This will only work if Ollama service is running')
    # Uncomment to run
    # result_ollama = prompt_ollama(test_text, model='llama3.2:1b')
    # print('Result:', result_ollama[:300] if result_ollama else 'Failed')
    print('(Skipped - Ollama service must be running)')

## Section 8: Fine-Tuning with Transformers (Local Model)

**STEP (8)**: Fine-tune a local model using PEFT (Parameter-Efficient Fine-Tuning).

⚠️ **WARNING**: This step requires a **high-end GPU or server** with significant VRAM (24GB+).
   - For consumer GPUs (e.g., RTX 3090), reduce batch size or skip this section
   - Recommended: A100, H100, or similar enterprise GPUs
   - If resources unavailable, set `SKIP_FINETUNING=True` below

In [ ]:
# Set this to True if you want to skip fine-tuning (e.g., insufficient GPU memory)
SKIP_FINETUNING = True  # Change to False if you have adequate GPU resources

if SKIP_FINETUNING:
    print('⚠️  Fine-tuning section SKIPPED')
    print('Reason: GPU resource intensive (requires 24GB+ VRAM)')
    print('To enable: Set SKIP_FINETUNING=False if you have a high-end GPU/server')
else:
    print('Fine-tuning will proceed (ensure adequate GPU resources available)')

In [ ]:
if not SKIP_FINETUNING:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from transformers import TrainingArguments, Trainer
    from peft import LoraConfig, TaskType, get_peft_model
    from datasets import Dataset
    import torch
    
    def prepare_finetuning_data_transformers(dataset, train_ratio=0.8, seed=1):
        train_df = dataset.sample(frac=train_ratio, random_state=seed)
        test_df = dataset.drop(index=train_df.index)
        return train_df, test_df
    
    # Prepare data
    train_df_transformers, test_df_transformers = prepare_finetuning_data_transformers(dataset)
    print(f'Data prepared for Transformers fine-tuning')
    print(f'Train: {len(train_df_transformers)}, Test: {len(test_df_transformers)}')
else:
    print('Transformers fine-tuning skipped as configured')

## Section 9: RAG with Ollama (Local Model)

**STEP (9)**: Advanced RAG implementation with local Ollama model.

⚠️ **WARNING**: This requires **advanced Ollama knowledge**:
   - Vector store setup (SKLearnVectorStore)
   - Embedding models (nomic-embed-text)
   - LangChain integration
   - Document chunking and retrieval optimization

If unfamiliar with these concepts, set `SKIP_RAG_OLLAMA=True` below

In [ ]:
# Set to True if you want to skip RAG with Ollama (advanced setup required)
SKIP_RAG_OLLAMA = True  # Change to False if you understand RAG/embedding workflows

if SKIP_RAG_OLLAMA:
    print('⚠️  RAG with Ollama section SKIPPED')
    print('Reason: Requires advanced knowledge of embedding models and vector stores')
    print('To enable: Set SKIP_RAG_OLLAMA=False if you understand RAG workflows')
else:
    print('RAG with Ollama will proceed')

In [ ]:
if not SKIP_RAG_OLLAMA and ollama_available:
    from langchain_core.documents import Document
    from langchain.text_splitter import RecursiveCharacterTextSplitter
    from langchain_community.vectorstores import SKLearnVectorStore
    from langchain_ollama import OllamaEmbeddings
    
    def setup_rag_ollama():
        # Create documents from context
        docs = [Document(page_content=rag_context)]
        
        # Split documents
        text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
            chunk_size=1000, chunk_overlap=0)
        doc_splits = text_splitter.split_documents(docs)
        
        # Create vector store
        vectorstore = SKLearnVectorStore.from_documents(
            documents=doc_splits,
            embedding=OllamaEmbeddings(model='nomic-embed-text'),
        )
        
        return vectorstore
    
    print('RAG setup for Ollama configured')
    print('Note: Requires nomic-embed-text model pulled in Ollama')
else:
    print('RAG with Ollama skipped')

## Section 10: Evaluation Framework

**STEP (10)**: Comprehensive evaluation using token-level metrics.

Import evaluation functions and compute Precision, Recall, Accuracy, and F1.

In [ ]:
# Import evaluation module
import sys
sys.path.insert(0, '.')

# Note: Assumes evaluation.py is in the same directory
try:
    from evaluation import do_praf
    print('✓ Evaluation module loaded')
    eval_available = True
except ImportError as e:
    print(f'⚠️  Could not import evaluation module: {e}')
    eval_available = False

In [ ]:
def parse_metaphor_tags(xml_text, tag_name='Metaphor'):
    """
    Extract text wrapped in metaphor tags.
    """
    pattern = f'<{tag_name}>(.*?)</{tag_name}>'
    matches = re.findall(pattern, xml_text, re.DOTALL)
    return matches

# Example evaluation on first entry
if eval_available:
    example_true = dataset.iloc[0]['metaphor_tagged_text']
    # For demo, we'll use the same as prediction (perfect match)
    example_pred = example_true
    
    print('\nEvaluation Example:')
    print(f'True tags found: {len(parse_metaphor_tags(example_true))}')
    print(f'Pred tags found: {len(parse_metaphor_tags(example_pred))}')
else:
    print('Evaluation module not available')

## Section 11: Full Dataset Evaluation

**STEP (11)**: Compute comprehensive metrics across the entire dataset.

In [ ]:
def evaluate_all_samples(dataset, predictions=None):
    """
    Evaluate metaphor identification across all samples.
    
    Args:
        dataset: DataFrame with ground truth
        predictions: List of predictions (or use ground truth for perfect match)
    
    Returns:
        Dictionary with metrics
    """
    if predictions is None:
        # Use ground truth as prediction for demonstration
        predictions = dataset['metaphor_tagged_text'].tolist()
    
    all_results = []
    
    for idx in range(len(dataset)):
        xml_true = dataset.iloc[idx]['metaphor_tagged_text']
        xml_pred = predictions[idx]
        
        if eval_available:
            try:
                result = do_praf(xml_true, xml_pred, 'Metaphor')
                all_results.append(result)
            except Exception as e:
                print(f'Error evaluating sample {idx}: {e}')
                continue
    
    if not all_results:
        print('No valid results to aggregate')
        return None
    
    # Aggregate metrics
    avg_precision = np.mean([r['precision'] for r in all_results])
    avg_recall = np.mean([r['recall'] for r in all_results])
    avg_accuracy = np.mean([r['accuracy'] for r in all_results])
    avg_f1 = np.mean([r['f1'] for r in all_results])
    
    return {
        'precision': avg_precision,
        'recall': avg_recall,
        'accuracy': avg_accuracy,
        'f1': avg_f1,
        'num_samples': len(all_results)
    }

# Run evaluation
if eval_available:
    print('Running comprehensive evaluation on all samples...')
    evaluation_results = evaluate_all_samples(dataset)
    
    if evaluation_results:
        print('\n' + '='*60)
        print('COMPREHENSIVE EVALUATION RESULTS')
        print('='*60)
        print(f'Number of samples evaluated: {evaluation_results["num_samples"]}')
        print(f'\nAverage Precision: {evaluation_results["precision"]:.4f}')
        print(f'Average Recall:    {evaluation_results["recall"]:.4f}')
        print(f'Average Accuracy:  {evaluation_results["accuracy"]:.4f}')
        print(f'Average F1 Score:  {evaluation_results["f1"]:.4f}')
        print('='*60)
else:
    print('⚠️  Evaluation module not available - skipping evaluation')

## Summary

### Workflow Sections:

1. ✓ **Dataset Loader** - Loaded metaphor dataset (94 entries)
2. ✓ **Dataset Analysis** - Examined structure and statistics
3. ✓ **OpenAI Prompting** - Prompting-based metaphor identification
4. ✓ **OpenAI Fine-tuning** - Fine-tune models on OpenAI
5. ✓ **OpenAI RAG** - Retrieval-augmented generation with OpenAI
6. ✓ **Ollama Setup** - Installation and configuration guide
7. ✓ **Ollama Prompting** - Local model-based metaphor identification
8. ⚠️ **Transformers Fine-tuning** - Local fine-tuning (GPU-intensive, configurable)
9. ⚠️ **Ollama RAG** - Advanced RAG with local embeddings (optional, configurable)
10. ✓ **Evaluation Framework** - Token-level evaluation metrics
11. ✓ **Full Evaluation** - Comprehensive metrics on entire dataset

### Configuration Options:

- `SKIP_FINETUNING`: Set to `True` to skip resource-intensive fine-tuning
- `SKIP_RAG_OLLAMA`: Set to `True` to skip advanced RAG setup
- `my_api_key`: Set your OpenAI API key for cloud-based tasks

### Output Metrics:

- **Precision**: Correctly identified metaphors / Total predicted metaphors
- **Recall**: Correctly identified metaphors / Total actual metaphors
- **Accuracy**: Correctly classified tokens / Total tokens
- **F1 Score**: Harmonic mean of Precision and Recall